# 02 Features

PLAN.md step 2. Logic in `src/passmodel/features.py`; this notebook reads `data/processed/features.parquet` (written by `scripts/run_pipeline.py`).

One row per (fold, pass): the intended target, and the features that depend on it, are learned separately for each leave-one-match-out fold. `fold` = that fold's test match.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from passmodel.features import FEATURE_GROUPS, FEATURES
from passmodel.load import REPO_ROOT

f = pd.read_parquet(REPO_ROOT / "data" / "processed" / "features.parquet")
held_out = f[(f["fold"] == f["match_id"]) & ~f["is_set_piece"]]  # each pass once, as its own test match sees it
len(f), len(held_out)

## D3: does the intended target match the actual receiver? (completed open-play passes)

In [ ]:
pd.read_csv(REPO_ROOT / "results" / "d3_target_agreement.csv")

## Feature groups

In [ ]:
pd.Series(FEATURE_GROUPS)

## Completed vs failed: feature means (held-out rows, open play)

In [ ]:
held_out.groupby("completed")[FEATURES].mean().T.round(2)

## Distributions by outcome

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), ["pass_length", "passer_nearest_opp", "target_nearest_opp", "lane_min_opp_dist", "target_confidence", "passer_speed"]):
    for done, colour, label in [(True, "#1f5fbf", "completed"), (False, "#d62728", "failed")]:
        ax.hist(held_out.loc[held_out["completed"] == done, col].clip(upper=held_out[col].quantile(0.99)), bins=30,
                density=True, alpha=0.5, color=colour, label=label)
    ax.set_title(col)
axes[0, 0].legend()
fig.suptitle("How completed and failed passes differ at the moment of the pass")
fig.tight_layout()

## Missing values

In [ ]:
held_out[FEATURES].isna().sum()[lambda s: s > 0]